# Image-to-Image AI Enhancement and Compositing Workflow

This workflow uses **Stable Diffusion** to enhance a background image and combine it with a foreground using a mask.

---

1. **Environment Setup**  
   Required libraries are installed to enable AI image generation and processing.

2. **Device Selection**  
   The script automatically uses a **GPU** if available for faster computation, otherwise it falls back to **CPU**.

3. **Load and Prepare Images**  
   - The **foreground**, **background**, and **mask** images are loaded.  
   - Foreground and background are converted to **RGB**, and the mask to **grayscale**.  
   - All images are resized to the same resolution.  
   - The mask is slightly blurred for smooth blending.

4. **Setup Image-to-Image AI Model**  
   - The **Stable Diffusion v1.5 image-to-image pipeline** is loaded.   
   - Optional memory-efficient attention is enabled to speed up generation.

5. **Generate Enhanced Background**  
   - The background is modified according to a **text prompt** describing the desired scene.  
   - Parameters like **strength**, **guidance scale**, and **inference steps** control how much the AI changes the image and how closely it follows the prompt.  
   - The result is a refined, visually enhanced background.

6. **Final Composite**  
   - The foreground and AI-enhanced background are combined using the mask.  
   - White areas of the mask preserve the foreground; black areas show the background.  
   - The final composite image is saved.

7. **Outcome**  
   You get a **high-quality composite image** where the foreground blends naturally with a detailed, AI-enhanced background.

In [ ]:
from google.colab import files


print("Upload your full snapshot image and mask:")
uploaded = files.upload()
foreground_path = list(uploaded.keys())[0]
background_path = list(uploaded.keys())[1]
mask_path = list(uploaded.keys())[2]

In [ ]:
import cv2
import numpy as np


def create_mask(input_path, output_path):
    img = cv2.imread(input_path)
    mask = np.any(img > 10, axis=2).astype(np.uint8) * 255  # threshold = 10 to ignore compression noise
    mask = cv2.medianBlur(mask, 3)
    cv2.imwrite(output_path, mask)
    print(f"Saved mask to {output_path}")

create_mask(mask_path, mask_path)

In [ ]:
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance


def enhance_realism_whole_image(
    input_path,
    output_path,
    noise_strength=0.02,
    blur_radius=2,
    brightness_factor=1.05,
    contrast_factor=1.05
  ):
    img = Image.open(input_path).convert("RGB")
    img_np = np.array(img).astype(np.float32)/255.0

    noise = np.random.normal(0, noise_strength, img_np.shape)
    img_noisy = np.clip(img_np + noise, 0, 1)
    img_noisy = Image.fromarray((img_noisy*255).astype(np.uint8))

    soft = img_noisy.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    blended = Image.blend(img_noisy, soft, alpha=0.25)
    enhancer = ImageEnhance.Brightness(blended)
    blended = enhancer.enhance(brightness_factor)
    enhancer = ImageEnhance.Contrast(blended)
    blended = enhancer.enhance(contrast_factor)
    blended.save(output_path)
    print(f"Saved enhanced image to {output_path}")

enhance_realism_whole_image(
    input_path=foreground_path,
    output_path=foreground_path,
    noise_strength=0.02,
    blur_radius=2,
    brightness_factor=0.95,
    contrast_factor=0.95
)

In [ ]:
# Setup environment
!pip install -q diffusers transformers accelerate safetensors pillow torch torchvision xformers

import torch
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import make_image_grid, load_image
from PIL import Image, ImageOps, ImageFilter


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

foreground = Image.open(foreground_path).convert("RGB")
background = Image.open(background_path).convert("RGB")
mask = Image.open(mask_path).convert("L")

RESIZE_W, RESIZE_H = 1024, 1024
foreground = foreground.resize((RESIZE_W, RESIZE_H))
background = background.resize((RESIZE_W, RESIZE_H))
mask = mask.resize((RESIZE_W, RESIZE_H))
mask = mask.filter(ImageFilter.GaussianBlur(radius=2))

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)
pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()
# remove the line if xFormers is not installed or you have PyTorch 2.0 or higher installed

# Generate / enhance background
print("Generating enhanced background...")
background_prompt = "16th century Age of Discovery coastline of western Taiwan, sunset, sandbars and shallow coastline, mountain in the back, Portuguese ships anchoring in distance, warm tropical climate, cinematic realism"
seed = 9999
generator = torch.Generator(device=device).manual_seed(seed)
refined_background = pipe(
    prompt=background_prompt,
    image=background,
    strength=0.6,
    guidance_scale=8.0,
    num_inference_steps=50,
    generator=generator,
    width=RESIZE_W,
    height=RESIZE_H,
).images[0]
refined_background.save("refined_background.png")

# Final composite
print("Generating composite...")
result = Image.composite(foreground, refined_background, mask)
result.save("final_composite.png")
print("Complete")
